# HNSCC-3DCT-RT

Bejarano, Tatiana, Mariluz De Ornelas‐Couto, and Ivaylo B. Mihaylov. "Longitudinal fan‐beam computed tomography dataset for head‐and‐neck squamous cell carcinoma patients."  Medical physics 46.5 (2019): 2526-2537.

"Data were exported from Eclipse in DICOM (images), DICOM-RTSTRUCT (structures), and DICOM-RTDOSE (doses) format and then imported into MIM. It should be noted that we have two DICOM-RTst per imaging set, one with series description MIM structures and another named ARIA Rad Onc Structure Sets. The RTst-MIM structures was renamed by us to remain consistent with Table V structures. On the other hand, RTst-ARIA Rad Onc Structure Sets were the original structures as part of treatment planning and delivery. Our recommendation is to use RTst-MIM structures so that the information on Table V is helpful."

In [ ]:
"""
CT and RTDOSE files can be matched based on the DICOM tag (0020,0052) Frame of Reference UID
CT and RTSTRUCT files can be matched based on the DICOM tag (0020,000d) Study Instance UID + (0008,1090) Manufacturer's Model Name
"""

In [ ]:
import pickle
with open(r"C:\Users\bilel.guetarni\Desktop\SEQ-RT\data\tcia.pkl", "rb") as f:
    patients = pickle.load(f)

In [ ]:
import pydicom

txt = ""
for p in patients:
    p.sort_imaging()
    ct = p.ct[0]
    txt += str(p.id)
    if not(ct.rtstruct is None):
        dcm = pydicom.dcmread(ct.rtstruct.get_dcm_path())
        for roi in dcm.StructureSetROISequence:
            txt += f"\n\t{roi.ROIName}"
    txt += "\n\n"

with open(r"C:\Users\bilel.guetarni\Desktop\tmp\oars.txt", "w") as file:
    file.write(txt)

# plot data statistics

In [52]:
import numpy as np
import plotly.express as px
import plotly.graph_objs as go

def plot_clinical(hpv=None, lrr=None, os=None):
    if not hpv is None:
        fig = go.Figure(data=[go.Pie(labels=hpv.unique(), values=[list(hpv.values).count(k) for k in hpv.unique()], title="HPV")])
        fig.show()

    if not lrr is None:
        fig = go.Figure(data=[go.Pie(labels=lrr.unique(), values=[list(lrr.values).count(k) for k in lrr.unique()], title="locoregional recurrence")])
        fig.show()

    if not os is None:
        x = np.sort(os.values.astype(np.int16))
        y = [x[x >= i].size/x.size for i in x]
        fig = px.line(x=x, y=y, title="overall survival")
        fig.show()

## name
HPV
Local + Regional
Distant Metastasis
RFS
OS

## Head-Neck-CT-Atlas
/
Loco-regional Control Censor ; Site of recurrence (Distal/Local/ Locoregional)
Site of recurrence (Distal/Local/ Locoregional)
Disease-free interval (months)
Survival  (months)

## Head-Neck-PET-CT
HPV status
Locoregional
Distant
Time – diagnosis to LR (days) ; Time – diagnosis to DM (days)
Time – diagnosis to Death (days)

## HNSCC-3DCT-RT
/
/
/
/
/

## Oropharyngeal-Radiomics-Outcomes
HPV Status
Locoregional control
Freedom from distant metastasis
Relapse-free survival
Overall survival_duration of Merged updated ASRM V2

## QIN-HEADNECK
/
Location of First Recurrence
Location of First Recurrence
Date of Recurrence
Date of Death

## RADCURE
HPV
Local ; Regional
Distant
Date Local ; Date Regional ; Date Distant
Date of Death

# HECKTOR 
HPV Status
Relapse
Relapse
RFS
/

In [ ]:
import pandas

# Head-Neck-CT-Atlas
df = pandas.read_excel(r"F:\TCIA\Head-Neck-CT-Atlas\HNSCC-MDA-Data_update_20240514.xlsx", sheet_name="HNSCC-MDA-Data_update")
print(df.columns)

# "Disease-free interval (months)"
lrr = df["Loco-regional Control Censor"]
lrr = lrr[lrr.notna()]
lrr[lrr == 1] = "negative"
lrr[lrr == 0] = "positive"

c = "Survival  (months)"
os = df[df[c].notna()][c]
os *= 30

plot_clinical(lrr=lrr, os=os)

Index(['TCIA PatientID', 'Sex', 'Age', 'Date of Birth', 'Diag', 'Site',
       'Histology', 'Grade', 'T', 'N', 'M', 'Stage',
       'Offset Date of Diagnosis', 'Offset Last Contact Date',
       'Follow up duration (day)', 'Follow up duration (year)',
       'Follow up duration (month)', 'Offset Date of Death',
       'Survival  (months)', 'Alive or Dead', 'Cause of Death',
       'Offset Date of recurrence', 'Disease-free interval (months)',
       'Site of recurrence (Distal/Local/ Locoregional)',
       'Overall Survival Censor', 'Disease Specific Survival Censor',
       'Loco-regional Control Censor', 'Oncologic Treatment Summary',
       'Induction Chemotherapy', 'Chemotherapy Regimen',
       'Platinum-based chemotherapy', 'Received Concurrent Chemoradiotherapy?',
       'CCRT Chemotherapy Regimen', 'Surgery Summary', 'RT Total Dose (Gy)',
       'Dose/Fraction (Gy/fx)', 'Number of Fractions',
       'Unplanned Additional Oncologic Treatment', 'Smoking History',
       'Current 

C:\Users\bilel.guetarni\AppData\Local\Temp\3\ipykernel_12260\1747623523.py:7: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'negative' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.



In [ ]:
import pandas

# Head-Neck-PET-CT
df = []
for i in ("HGJ", "CHUS", "HMR", "CHUM"):
    df__ = pandas.read_excel(r"F:\TCIA\Head-Neck-PET-CT\INFOclinical_HN_Version2_30may2018.xlsx", sheet_name=i)
    df__["center"] = i
    df.append(df__)
df = pandas.concat(df)
print(df.columns)

hpv = df["HPV status"]
hpv[hpv == 'N/A'] = float("nan")
hpv[hpv == '-'] = "negative"
hpv[hpv == '+'] = "positive"

# "Time – diagnosis to LR (days)"
lrr = df['Locoregional']

c = "Time – diagnosis to Death (days)"
os = df[df[c].notna()][c]

plot_clinical(hpv=hpv, lrr=lrr, os=os)

Index(['Patient #', 'Sex', 'Age', 'Primary Site', 'T-stage', 'N-stage',
       'M-stage', 'TNM group stage', 'HPV status',
       'Time – diagnosis to diagnosis (days)',
       'Time – diagnosis to PET (days)', 'Time – diagnosis to CT sim (days)',
       'Time – diagnosis to start treatment (days)',
       'Time – diagnosis to end treatment (days)', 'Therapy', 'Surgery',
       'Time – diagnosis to last follow-up(days)', 'Locoregional', 'Distant',
       'Death', 'Time – diagnosis to LR (days)',
       'Time – diagnosis to DM (days)', 'Time – diagnosis to Death (days)',
       'Time – diagnosis to last follow-up (days)'],
      dtype='object')


C:\Users\bilel.guetarni\AppData\Local\Temp\3\ipykernel_12260\3693279130.py:12: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\bilel.guetarni\AppData\Local\Temp\3\ipykernel_12260\3693279130.py:13: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\bilel.guetarni\AppData\Local\Temp\3\ipykernel_12260\3693279130.py:14: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
import pandas

# Oropharyngeal-Radiomics-Outcomes
df = pandas.read_csv(r"F:\TCIA\Oropharyngeal-Radiomics-Outcomes\Radiomics_Outcome_Prediction_in_OPC_ASRM_corrected.csv")
print(df.columns)

hpv = df["HPV Status"]
hpv[hpv == 'Unknown'] = float("nan")
hpv[hpv == 'N'] = "negative"
hpv[hpv == 'P'] = "positive"

# "Locoregional control_duration of Merged updated ASRM V2"
lrr = df['Locoregional control']

c = "Overall survival_duration of Merged updated ASRM V2"
os = df[df[c].notna()][c]

plot_clinical(hpv=hpv, lrr=lrr, os=os)

Index(['TCIA Radiomics dummy ID of To_Submit_Final', 'Gender', 'Age at Diag',
       'Smoking status', 'Smoking status (Packs-Years)', 'Tumor laterality',
       'Cancer subsite of origin', 'HPV Status', 'T-category', 'N-category',
       'AJCC Stage (7th edition)', 'Therapeutic Combination',
       'Radiation Treatment_duration',
       'Total prescribed Radiation treatment dose',
       'Radiation treatment_number of fractions',
       'Radiation treatment_dose per fraction', 'Vital status',
       'Overall survival_duration of Merged updated ASRM V2', 'Local control',
       'Local control_duration of Merged updated ASRM V2', 'Regional control',
       'Regional control_duration of Merged updated ASRM V2',
       'Locoregional control',
       'Locoregional control_duration of Merged updated ASRM V2',
       'Freedom from distant metastasis',
       'Freedom from distant metastasis_duration of Merged updated ASRM V2',
       'Relapse-free survival', 'Days to last FU',
       'Neck D

C:\Users\bilel.guetarni\AppData\Local\Temp\3\ipykernel_12260\3683036075.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hpv[hpv == 'Unknown'] = float("nan")
C:\Users\bilel.guetarni\AppData\Local\Temp\3\ipykernel_12260\3683036075.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hpv[hpv == 'N'] = "negative"
C:\Users\bilel.guetarni\AppData\Local\Temp\3\ipykernel_12260\3683036075.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  

In [ ]:
import pandas
import numpy as np

def convert_to_datetime(df, c):
    return pandas.to_datetime(df[c], format="%Y-%m-%d %H:%M:%S")

# QIN-HEADNECK
df1 = pandas.read_excel(r"F:\TCIA\QIN-HEADNECK\Batch_01-and-Batch_02-Clinical-Data_aug242020.xlsx", header=0, sheet_name="uiowa_clinical_data_batch1_aug2")
df2 = pandas.read_excel(r"F:\TCIA\QIN-HEADNECK\Batch_01-and-Batch_02-Clinical-Data_aug242020.xlsx", header=0, sheet_name="uiowa_clinical_data_batch2_aug2")
df = pandas.concat((df1, df2)).drop(index=0)
print(df.columns)

# "Date of cancer recurrence"
lrr_valid = ['Regional', 'Local', 'Local/Distant', 'Locoregional', 'Both']
lrr = df['Location of first recurrence']
lrr = lrr[lrr.isin(lrr_valid)]

c1, c2 = "Radiotherapy Procedure, Date treatment stopped", "Date of death"
os = df[[c1, c2]]
os = os.dropna()
os[c1] = convert_to_datetime(os, c1)
os[c2] = convert_to_datetime(os, c2)
os["os"] = [(row[c2] - row[c1]).days for _, row in os.iterrows()]
os = os["os"]

plot_clinical(lrr=lrr, os=os)

Index(['PatientID', 'Patient's Birth Date', 'Patient's Sex',
       'Patient's Weight', 'Patient's Height', 'Patient's Race', 'Hispanic',
       'History of Diabetes Mellitus', 'History of radiation therapy',
       'History of malignant neoplasm',
       ...
       'Date of 2nd primary', 'Date of cancer recurrence',
       'Location of first recurrence',
       'Diagnositic Procedure, Biopsy, Date of procedure.4', 'Biopsy site.4',
       'Chemotherapy, Date treatment started.2',
       'Chemotherapy, Date treatment stopped.2',
       'Chemotherapy, Antineoplastic agent.6',
       'Chemotherapy, Antineoplastic agent.7',
       'Chemotherapy, Antineoplastic agent.8'],
      dtype='object', length=104)


In [57]:
import pandas

def convert_to_datetime(df, c):
    return pandas.to_datetime(df[c], format="%d/%m/%Y")

# RADCURE
df = pandas.read_excel(r"F:\TCIA\RADCURE\RADCURE_Clinical_v04_20241219 (1).xlsx")
print(df.columns)

hpv = df["HPV"]
hpv[hpv == 'Yes, Negative'] = "negative"
hpv[hpv == 'Yes, positive'] = "positive"


# lrr_valid = ['Persistent', 'Yes', 'Possible']
# lrr = df[["Local", "Regional", "Distant"]]
# lrr = lrr[lrr.isin(lrr_valid)]
# print(lrr)

c1, c2 = "RT Start", "Date of Death"
os = df[[c1, c2]]
os = os.dropna()
os[c1] = convert_to_datetime(os, c1)
os[c2] = convert_to_datetime(os, c2)
os["os"] = [(row[c2] - row[c1]).days for _, row in os.iterrows()]
os = os["os"]

# plot_clinical(hpv=hpv, lrr=lrr, os=os)
plot_clinical(hpv=hpv, os=os)

Index(['patient_id', 'Age', 'Sex', 'ECOG PS', 'Smoking PY', 'Smoking Status',
       'Ds Site', 'Subsite', 'T', 'N', 'M ', 'Stage', 'Path', 'HPV',
       'Tx Modality', 'Chemo', 'RT Start', 'Dose', 'Fx', 'Last FU', 'Status',
       'Length FU', 'Date of Death', 'Cause of Death', 'Local', 'Date Local',
       'Regional', 'Date Regional', 'Distant', 'Date Distant', '2nd Ca',
       'Date 2nd Ca', 'RADCURE-challenge', 'ContrastEnhanced'],
      dtype='object')


C:\Users\bilel.guetarni\AppData\Local\Temp\3\ipykernel_2752\422800019.py:11: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\bilel.guetarni\AppData\Local\Temp\3\ipykernel_2752\422800019.py:12: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [38]:
import pandas

# HECKTOR 
df = pandas.read_csv(r"E:\bilel\HECKTOR 2025 Training Data\Task 3\HECKTOR_2025_Training_Task_3.csv")
print(df.columns)

hpv = df["HPV Status"]
hpv[hpv == 1] = "positive"
hpv[hpv == 0] = "negative"

df = pandas.read_csv(r"E:\bilel\HECKTOR 2025 Training Data\Task 2\HECKTOR_2025_Training_Task_2.csv")
print(df.columns)

# "In particular, local, regional, and distant metastases are events and all others are censored. 
# Time to event, defined in days, starts with the end of radiation therapy."
lrr = df["Relapse"]
os = df["RFS"]

plot_clinical(hpv=hpv, lrr=lrr, os=os)

Index(['PatientID', 'CenterID', 'Age', 'Gender', 'Tobacco Consumption',
       'Alcohol Consumption', 'Performance Status', 'Treatment', 'M-stage',
       'HPV Status'],
      dtype='object')
Index(['PatientID', 'CenterID', 'Age', 'Gender', 'Tobacco Consumption',
       'Alcohol Consumption', 'Performance Status', 'Treatment', 'M-stage',
       'Relapse', 'RFS'],
      dtype='object')


C:\Users\bilel.guetarni\AppData\Local\Temp\3\ipykernel_2752\634278530.py:8: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\bilel.guetarni\AppData\Local\Temp\3\ipykernel_2752\634278530.py:8: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'positive' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.



# center ID

In [ ]:
# Oropharyngeal-Radiomics-Outcomes (1)
#   M.D. Anderson Cancer Center (MDA)

# Head-Neck-CT-Atlas (1)
#   MD Anderson Cancer Center (MDA)

# QIN-HEADNECK (1)
#   University of Iowa Hospital and Clinics (UIHC)

# Head-Neck-PET-CT (4)
#   HGJ: Hôpital général juif
#   CHUS: Centre hospitalier universitaire de Sherbrooke
#   HMR: Hôpital Maisonneuve-Rosemont de Montréal
#   CHUM: Centre hospitalier de l’Université de Montréal

# RADCURE (1)
#   University Health Network (UHN)

# HNSCC-3DCT-RT (1)
#   University of Miami Miller School of Medicine (UMMSM)

# HECKTOR Challenge 2025 (10)
#   HGJ: Hôpital général juif
#   CHUS: Centre hospitalier universitaire de Sherbooke
#   HMR: Hôpital Maisonneuve-Rosemont
#   CHUM: Centre hospitalier de l’Université de Montréal
#   CHUP: Centre Hospitalier Universitaire de Poitiers
#   MDA: MD Anderson Cancer Center
#   USZ: UniversitätsSpital Zürich
#   CHB: Centre Henri Becquerel
#   CHUB: Centre Hospitalier Universitaire de Brest
#   CHUN: Centre Hospitalier Universitaire de Nantes

# manufacturer & machine 

In [48]:
import pydicom
import os, glob
import tqdm
import pandas

def get_dicom_folder(path_):
    if not os.path.isdir(path_):
        return []
    
    paths_return = []

    list_of_path = glob.glob(os.path.join(path_, "*"))
    if all(map(lambda i: i.endswith(".dcm"), list_of_path)):
        return [path_]
    else:
        for i in list_of_path:
            paths_return.extend(get_dicom_folder(i))
    
    # print(f" found {len(paths_return)} dicom folders")
    return paths_return

def plot_machine_dist(path_):
    df = []
    for dcm_path in tqdm.tqdm(get_dicom_folder(path_)):
        dcm_path = os.path.join(dcm_path, os.listdir(dcm_path)[0])
        dcm = pydicom.dcmread(dcm_path)

        try:
            modality = dcm.get((0x0008, 0x0060)).value
            if not modality == "CT":
                continue

            df.append({"manufacturer": dcm.get((0x0008, 0x0070)).value, "model name": dcm.get((0x0008, 0x1090)).value})
        except (PermissionError, AttributeError, TypeError):
            continue

    df = pandas.DataFrame(df)

    fig = go.Figure(data=[go.Pie(
        labels=df["manufacturer"].unique(), 
        values=[list(df["manufacturer"]).count(k) for k in df["manufacturer"].unique()],
        title="manufacturer")])
    fig.show()

    fig = go.Figure(data=[go.Pie(
        labels=df["model name"].unique(), 
        values=[list(df["model name"]).count(k) for k in df["model name"].unique()],
        title="model")])
    fig.show()

In [32]:
plot_machine_dist(r"F:\TCIA\Head-Neck-CT-Atlas")

100%|██████████| 3222/3222 [01:41<00:00, 31.63it/s]


In [49]:
plot_machine_dist(r"F:\TCIA\Head-Neck-PET-CT")

100%|██████████| 2661/2661 [14:10<00:00,  3.13it/s]


In [33]:
plot_machine_dist(r"F:\TCIA\HNSCC-3DCT-RT")

100%|██████████| 371/371 [02:10<00:00,  2.84it/s]


In [30]:
plot_machine_dist(r"F:\TCIA\Oropharyngeal-Radiomics-Outcomes")

100%|██████████| 814/814 [00:24<00:00, 33.57it/s]

           manufacturer          model name
0                   001      LightSpeed VCT
1                   001        LightSpeed16
2    GE MEDICAL SYSTEMS        LightSpeed16
3                   001        LightSpeed16
4                   001        LightSpeed16
..                  ...                 ...
407                 001      LightSpeed VCT
408                 001  Discovery CT750 HD
409                 004            Aquilion
410                 001      LightSpeed VCT
411  GE MEDICAL SYSTEMS      LightSpeed VCT

[412 rows x 2 columns]


In [34]:
plot_machine_dist(r"F:\TCIA\QIN-HEADNECK")

100%|██████████| 3837/3837 [04:50<00:00, 13.23it/s]


In [36]:
plot_machine_dist(r"F:\TCIA\RADCURE")

100%|██████████| 6683/6683 [13:44<00:00,  8.10it/s]


In [143]:
from pydicom import dcmread
ds = dcmread(r"E:\bilel\ARTIX\ARTIX\DICOM_ARTIX_data\015\CT3\DOE^JOHN_ANON95093_CT_2014-07-21_121416_ORL.(sauf.sinus)_ORL.2MM_n209__00000\2.16.840.1.114362.1.11956109.23961857828.605405526.805.34.dcm")
for element in ds:
    print(element)

(0008,0005) Specific Character Set              CS: 'ISO_IR 100'
(0008,0008) Image Type                          CS: ['DERIVED', 'SECONDARY', 'AXIAL']
(0008,0012) Instance Creation Date              DA: '20140721'
(0008,0013) Instance Creation Time              TM: '121653'
(0008,0016) SOP Class UID                       UI: CT Image Storage
(0008,0018) SOP Instance UID                    UI: 2.16.840.1.114362.1.11956109.23961857828.605405526.805.34
(0008,0020) Study Date                          DA: '20140721'
(0008,0022) Acquisition Date                    DA: '20140721'
(0008,0023) Content Date                        DA: '20140721'
(0008,0030) Study Time                          TM: '121416'
(0008,0032) Acquisition Time                    TM: '121556'
(0008,0033) Content Time                        TM: '121556.880001'
(0008,0050) Accession Number                    SH: ''
(0008,0060) Modality                            CS: 'CT'
(0008,0070) Manufacturer                        LO: 'Ph

In [159]:
import os, glob, pathlib
from pydicom import dcmread

def find_all_dcm_folders(path):
    data = []
    try:
        if all([pydicom.misc.is_dicom(j) for j in glob.glob(os.path.join(path, "*"))]):
            return [path]
    except PermissionError:
        pass

    for folder in glob.glob(os.path.join(path, "*")):
        if os.path.isdir(folder):
            data.extend(find_all_dcm_folders(folder))

    return data

dcm_folder = find_all_dcm_folders(r"F:\TCIA\Head-Neck-CT-Atlas\manifest-42ozNKaA8941017083770104413\HNSCC\HNSCC-01-0126")
print(len(dcm_folder))

for j in dcm_folder:
    print(j)
    ds = dcmread(os.path.join(j, os.listdir(j)[0]))
    try:
        print("mod: ", ds.get((0x0008, 0x0060)).value)
        print("study", ds[0x0020,0x000d].value)
        print("frame", ds[0x0020,0x0052].value)
    except (KeyError, AttributeError):
        pass
    
    print("\n")

14
F:\TCIA\Head-Neck-CT-Atlas\manifest-42ozNKaA8941017083770104413\HNSCC\HNSCC-01-0126\03-05-2000-NA-PET CT THYROID CANCER-25282\11.000000-PETNACHEAD-50684
mod:  PT
study 1.3.6.1.4.1.14519.5.2.1.1706.8040.144907355617929983729851525282
frame 1.3.6.1.4.1.14519.5.2.1.1706.8040.256816815614555553276785400296


F:\TCIA\Head-Neck-CT-Atlas\manifest-42ozNKaA8941017083770104413\HNSCC\HNSCC-01-0126\03-05-2000-NA-PET CT THYROID CANCER-25282\10.000000-PETACHEAD-72205
mod:  PT
study 1.3.6.1.4.1.14519.5.2.1.1706.8040.144907355617929983729851525282
frame 1.3.6.1.4.1.14519.5.2.1.1706.8040.256816815614555553276785400296


F:\TCIA\Head-Neck-CT-Atlas\manifest-42ozNKaA8941017083770104413\HNSCC\HNSCC-01-0126\03-05-2000-NA-PET CT THYROID CANCER-25282\4.000000-PETAC-15825
mod:  PT
study 1.3.6.1.4.1.14519.5.2.1.1706.8040.144907355617929983729851525282
frame 1.3.6.1.4.1.14519.5.2.1.1706.8040.271174204674460296844962256290


F:\TCIA\Head-Neck-CT-Atlas\manifest-42ozNKaA8941017083770104413\HNSCC\HNSCC-01-0126\03